# 09.8 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/08-prompt-engineering/exercises.ipynb)

Answers in [solutions.ipynb](solutions.ipynb).

In [ ]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

In [ ]:
sys.path.insert(0, str(repo_file("lib/course_llm.py").parent))
import course_llm
from course_llm import ask, ask_json, chat, info
info();

In [ ]:
# 20 real-looking ShopWave support emails, labelled by the support team.
EMAILS = [
 ("Hi, I ordered on Monday and the tracking still says 'label created'. When will it arrive?", "delivery"),
 ("My parcel was supposed to come yesterday with express but nothing showed up. Order 58213.", "delivery"),
 ("Do you deliver to Norway, and how long does it take?", "delivery"),
 ("The courier left a card saying they'll try again. Can I pick it up somewhere instead?", "delivery"),
 ("I'd like to send back the linen cushion, it doesn't match my sofa. How do I return it?", "returns_refunds"),
 ("I returned two candles 10 days ago and still haven't seen the money.", "returns_refunds"),
 ("Can I swap the blue throw for the grey one?", "returns_refunds"),
 ("You refunded £24 but I paid £27.95 including delivery. Where's the rest?", "returns_refunds"),
 ("Three of the six wine glasses arrived smashed. Photos attached.", "damaged_item"),
 ("The lamp flickers and then cuts out after a few minutes. It's only two weeks old.", "damaged_item"),
 ("I ordered the large planter but you sent the small one.", "damaged_item"),
 ("The box was soaked and the tea towels inside are stained.", "damaged_item"),
 ("We run a gift shop in York and would like to stock your candles. What are your trade prices?", "wholesale"),
 ("Our café account has had 3 orders now. Can we move to invoice payment?", "wholesale"),
 ("Is there a minimum quantity for wholesale orders of the ceramic mugs?", "wholesale"),
 ("I need 40 matching mugs for our office opening, can we get a business discount?", "wholesale"),
 ("Just wanted to say the fairy lights are gorgeous, thank you!", "other"),
 ("How many Stars do I have and when do they expire?", "other"),
 ("Is the reed diffuser safe to use around cats?", "other"),
 ("Can I change the email address on my account?", "other"),
]
LABELS = ["delivery", "returns_refunds", "damaged_item", "wholesale", "other"]
print(len(EMAILS), "emails;", {l: sum(lab == l for _, lab in EMAILS) for l in LABELS})

In [ ]:
SCHEMA = {"type": "object",
          "properties": {"category": {"type": "string", "enum": LABELS}},
          "required": ["category"], "additionalProperties": False}

import json

def classify(prompt, system=None):
    """One label, at temperature 0 so the same email always gets the same answer."""
    reply = chat([{"role": "user", "content": prompt}], system=system,
                 max_tokens=50, temperature=0, schema=SCHEMA)
    return json.loads(reply)["category"]

def evaluate(make_prompt, system=None):
    """Classify all 20 emails; return accuracy and the mistakes."""
    mistakes = []
    for text, label in EMAILS:
        predicted = classify(make_prompt(text), system)
        if predicted != label:
            mistakes.append((text[:55], label, predicted))
    return 1 - len(mistakes) / len(EMAILS), mistakes

In [ ]:
FEW_SHOT_SYSTEM = """You sort ShopWave support emails into exactly one category.

Categories:
- delivery: where an order is, delivery times, couriers, shipping destinations
- returns_refunds: sending items back, exchanges, refund amounts and timing
- damaged_item: broken, faulty, stained or wrong items received
- wholesale: businesses buying in bulk, trade accounts, trade prices, invoices
- other: anything else (praise, product questions, accounts, loyalty Stars)

Examples:
Email: "It's been 6 days and my order hasn't shipped." -> delivery
Email: "The vase you sent has a big crack down the side." -> damaged_item
Email: "Can I get my money back for the doormat? I changed my mind." -> returns_refunds
Email: "We own two florists and want to order your vases monthly." -> wholesale
Email: "Do your candles contain paraffin?" -> other"""

## Exercise 1: definitions without examples

Write a zero-shot prompt that includes a one-line definition of each category but **no examples**. Measure its accuracy on the 20 emails. Is it closer to plain zero-shot or to few-shot? What does that tell you about where the gain came from?

In [ ]:
# your code here

## Exercise 2: harder emails

Write five new emails that you think are genuinely hard to classify (mixed topics, sarcasm, very short). Label them yourself, then run the few-shot classifier. Which does it get wrong, and would a person agree with your label?

In [ ]:
# your code here

## Exercise 3: extract fields, not just a label

Design a JSON schema with `order_number` (string or empty), `product` (string), `issue` (one of: late, damaged, wrong_item, refund_amount, other) and `urgent` (boolean). Run it with `ask_json` on emails 1, 7, 8 and 10. Which fields are reliable?

In [ ]:
# your code here

## Exercise 4: fix a bad prompt (on paper first)

This prompt is used to write product descriptions: *"write about the candle"*. List at least four problems with it using the checklist, write a better version, then compare the two outputs for a "fig and cedar scented candle, 200 g, 40-hour burn".

In [ ]:
# your code here